In [51]:
import math
import random
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

SEED = 42
K = 10
NUM_NEGATIVES = 99
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_DIR = Path.cwd()

print('DEVICE =', DEVICE)


DEVICE = cuda


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
def build_id_mappings(values):
    uniq = pd.Index(sorted(pd.unique(values)))
    val2idx = {v: i for i, v in enumerate(uniq)}
    idx2val = {i: v for i, v in enumerate(uniq)}
    return uniq, val2idx, idx2val


def encode_series(series, mapping):
    return series.map(mapping).astype(np.int64)


def make_random_timestamps_for_lastfm(df, seed=42):
    rng = np.random.default_rng(seed)
    start_ts = 1262304000   # 2010-01-01
    end_ts   = 1609372800   # 2020-12-31
    df = df.copy()
    df['timestamp'] = rng.integers(start_ts, end_ts, size=len(df), endpoint=False)
    return df


def sample_99_negatives(all_items, user_full_seen, positive_item, rng, num_negatives=99):
    candidates = np.array(sorted(set(all_items) - set(user_full_seen) - {positive_item}), dtype=np.int64)
    if len(candidates) < num_negatives:
        return None
    negatives = rng.choice(candidates, size=num_negatives, replace=False)
    return np.concatenate([[positive_item], negatives])


def evaluate_sampled_ranking(score_fn, positive_items, candidate_items, k=10, batch_size=256):
    users = sorted(set(positive_items.keys()) & set(candidate_items.keys()))
    if len(users) == 0:
        return {f'HR@{k}': np.nan, f'NDCG@{k}': np.nan, 'MAP': np.nan, 'num_eval_users': 0}

    hr_total = 0.0
    ndcg_total = 0.0
    map_total = 0.0

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        score_matrix = score_fn(np.asarray(batch_users, dtype=np.int64))

        for row, u in enumerate(batch_users):
            pos_item = int(positive_items[u])
            cands = candidate_items[u]
            cand_scores = score_matrix[row, cands]
            order = np.argsort(-cand_scores)
            ranked_items = cands[order]
            rank = int(np.flatnonzero(ranked_items == pos_item)[0]) + 1  # 1-based rank

            hr_total += float(rank <= k)
            ndcg_total += float(1.0 / np.log2(rank + 1)) if rank <= k else 0.0
            map_total += float(1.0 / rank) 

    n = len(users)
    return {
        f'HR@{k}': hr_total / n,
        f'NDCG@{k}': ndcg_total / n,
        'MAP': map_total / n,
        'num_eval_users': n,
    }


In [ ]:
set_seed(SEED)

ml_ratings = pd.read_csv(BASE_DIR / 'ratings.csv')
ml_ratings = ml_ratings.rename(columns={
    'userId': 'user',
    'movieId': 'item',
    'rating': 'value',
    'timestamp': 'timestamp'
})

ml_ratings['user'] = ml_ratings['user'].astype(np.int64)
ml_ratings['item'] = ml_ratings['item'].astype(np.int64)
ml_ratings['value'] = ml_ratings['value'].astype(np.float32)
ml_ratings['timestamp'] = ml_ratings['timestamp'].astype(np.int64)
ml_ratings = ml_ratings.sort_values(['user', 'timestamp', 'item']).reset_index(drop=True)

RATING_THRESHOLD = 4.0
all_ml_items_raw = sorted(ml_ratings['item'].unique())

ml_train_parts = []
ml_test_positive_raw = {}
ml_user_full_seen_raw = {}
dropped_no_positive = 0
dropped_no_history_before_test = 0

for user, g in ml_ratings.groupby('user', sort=False):
    g = g.sort_values(['timestamp', 'item']).reset_index(drop=True)
    ml_user_full_seen_raw[int(user)] = set(g['item'].astype(int).tolist())

    pos_positions = np.flatnonzero((g['value'].to_numpy(dtype=np.float32) >= RATING_THRESHOLD))
    if len(pos_positions) == 0:
        dropped_no_positive += 1
        continue

    test_pos_loc = int(pos_positions[-1])
    if test_pos_loc == 0:
        dropped_no_history_before_test += 1
        continue

    train_part = g.iloc[:test_pos_loc].copy()
    test_row = g.iloc[test_pos_loc]

    ml_train_parts.append(train_part)
    ml_test_positive_raw[int(user)] = int(test_row['item'])

ml_train_raw = pd.concat(ml_train_parts, ignore_index=True)
known_ml_users = set(ml_train_raw['user'].unique())
known_ml_items = set(ml_train_raw['item'].unique())

valid_ml_users = []
for u, pos_item in ml_test_positive_raw.items():
    if u in known_ml_users and pos_item in known_ml_items:
        valid_ml_users.append(u)

ml_train = ml_train_raw[ml_train_raw['user'].isin(valid_ml_users)].copy()
ml_test_positive_raw = {u: ml_test_positive_raw[u] for u in valid_ml_users}
ml_user_full_seen_raw = {u: ml_user_full_seen_raw[u] for u in valid_ml_users}

ml_user_ids, ml_user2idx, ml_idx2user = build_id_mappings(ml_train['user'])
ml_item_ids, ml_item2idx, ml_idx2item = build_id_mappings(ml_train['item'])

ml_train['user_idx'] = encode_series(ml_train['user'], ml_user2idx)
ml_train['item_idx'] = encode_series(ml_train['item'], ml_item2idx)

ml_test_positive = {ml_user2idx[u]: ml_item2idx[i] for u, i in ml_test_positive_raw.items()}
ml_user_full_seen = {
    ml_user2idx[u]: {ml_item2idx[i] for i in items if i in ml_item2idx}
    for u, items in ml_user_full_seen_raw.items()
}

ml_all_items = np.arange(len(ml_item_ids), dtype=np.int64)
ml_rng = np.random.default_rng(SEED)
ml_candidate_items = {}
users_to_remove = []

for u_idx, pos_i_idx in ml_test_positive.items():
    sampled = sample_99_negatives(ml_all_items, ml_user_full_seen[u_idx], pos_i_idx, ml_rng, NUM_NEGATIVES)
    if sampled is None:
        users_to_remove.append(u_idx)
    else:
        ml_candidate_items[u_idx] = sampled

for u_idx in users_to_remove:
    ml_test_positive.pop(u_idx, None)

print('MovieLens train interactions =', len(ml_train))
print('MovieLens train users =', ml_train['user_idx'].nunique())
print('MovieLens train items =', ml_train['item_idx'].nunique())
print('Dropped users with no positive interaction =', dropped_no_positive)
print('Dropped users with no history before test positive =', dropped_no_history_before_test)
print('Final MovieLens eval users =', len(ml_test_positive))
ml_train.head()


MovieLens train interactions = 984305
MovieLens train users = 6037
MovieLens train items = 3703
Dropped users with no positive interaction = 2
Dropped users with no history before test positive = 0
Final MovieLens eval users = 6037


,user,item,value,timestamp,user_idx,item_idx
0,1,3186,4.0,978300019,0,2967
1,1,1022,5.0,978300055,0,956
2,1,1270,5.0,978300055,0,1177
3,1,1721,4.0,978300055,0,1573
4,1,2340,3.0,978300103,0,2145


## 3. MovieLens：Popularity baseline

In [ ]:
def ndcg_at_k_one_positive(ranked_items, positive_item, k=10):
    topk = ranked_items[:k]
    hits = np.where(topk == positive_item)[0]
    if len(hits) == 0:
        return 0.0
    rank = hits[0] + 1
    return 1.0 / np.log2(rank + 1)

def map_one_positive(ranked_items, positive_item):
    hits = np.where(ranked_items == positive_item)[0]
    if len(hits) == 0:
        return 0.0
    rank = hits[0] + 1
    return 1.0 / rank

def evaluate_popularity_scores(score_dict, model_name, default_score=None):
    hrs, ndcgs, maps = [], [], []

    if default_score is None:
        default_score = min(score_dict.values()) - 1.0 if len(score_dict) > 0 else -1e9

    for user, pos_item in test_positive_raw.items():
        candidates = ml_candidate_items[user]
        scores = np.array([score_dict.get(i, default_score) for i in candidates], dtype=np.float32)

        order = np.argsort(-scores, kind="stable")
        ranked_items = candidates[order]

        hr = 1.0 if pos_item in ranked_items[:K] else 0.0
        ndcg = ndcg_at_k_one_positive(ranked_items, pos_item, k=K)
        ap = map_one_positive(ranked_items, pos_item)

        hrs.append(hr)
        ndcgs.append(ndcg)
        maps.append(ap)

    result = pd.DataFrame([{
        "dataset": "MovieLens",
        "model": model_name,
        "HR@10": np.mean(hrs),
        "NDCG@10": np.mean(ndcgs),
        "MAP": np.mean(maps),
        "eval_users": len(hrs)
    }])
    return result

In [47]:
# Version 1: PosOnly Popularity
ml_train_pos_v1 = ml_train[ml_train["rating"] >= 4.0].copy()

item_pos_count_v1 = (
    ml_train_pos_v1.groupby("item")
    .size()
    .reset_index(name="score")
)

item_pos_count_v1 = item_pos_count_v1.sort_values(
    ["score", "item"],
    ascending=[False, True]
).reset_index(drop=True)

score_dict_v1 = dict(zip(item_pos_count_v1["item"], item_pos_count_v1["score"]))

result_v1 = evaluate_popularity_scores(
    score_dict=score_dict_v1,
    model_name="Popularity_V1_PosOnly"
)

result_v1

,dataset,model,HR@10,NDCG@10,MAP,eval_users
0,MovieLens,Popularity_V1_PosOnly,0.491716,0.271107,0.226227,6036


In [48]:
# Version 2: PosOnly + Time Decay
TIME_DECAY_LAMBDA = 0.001

ml_train_v2 = ml_train.copy()
max_ts_v2 = ml_train_v2["timestamp"].max()
ml_train_v2["age_days"] = (max_ts_v2 - ml_train_v2["timestamp"]) / 86400.0
ml_train_v2["time_weight"] = np.exp(-TIME_DECAY_LAMBDA * ml_train_v2["age_days"])

ml_train_pos_v2 = ml_train_v2[ml_train_v2["rating"] >= 4.0].copy()

item_pos_weight_v2 = (
    ml_train_pos_v2.groupby("item")["time_weight"]
    .sum()
    .reset_index(name="score")
)

item_pos_weight_v2 = item_pos_weight_v2.sort_values(
    ["score", "item"],
    ascending=[False, True]
).reset_index(drop=True)

score_dict_v2 = dict(zip(item_pos_weight_v2["item"], item_pos_weight_v2["score"]))

result_v2 = evaluate_popularity_scores(
    score_dict=score_dict_v2,
    model_name=f"Popularity_V2_PosOnly_TimeDecay_{TIME_DECAY_LAMBDA}"
)

result_v2

,dataset,model,HR@10,NDCG@10,MAP,eval_users
0,MovieLens,Popularity_V2_PosOnly_TimeDecay_0.001,0.494367,0.272676,0.227378,6036


In [49]:
# Version 3: PosOnly + Time Decay + Small-Sample Smoothing
TIME_DECAY_LAMBDA = 0.001
ALPHA = 10.0

ml_train_v3 = ml_train.copy()
max_ts_v3 = ml_train_v3["timestamp"].max()
ml_train_v3["age_days"] = (max_ts_v3 - ml_train_v3["timestamp"]) / 86400.0
ml_train_v3["time_weight"] = np.exp(-TIME_DECAY_LAMBDA * ml_train_v3["age_days"])

ml_train_pos_v3 = ml_train_v3[ml_train_v3["rating"] >= 4.0].copy()

item_stats_v3 = (
    ml_train_pos_v3.groupby("item")
    .agg(
        weighted_positive=("time_weight", "sum"),
        pos_count=("item", "size")
    )
    .reset_index()
)

item_stats_v3["score"] = (
    item_stats_v3["weighted_positive"] *
    (item_stats_v3["pos_count"] / (item_stats_v3["pos_count"] + ALPHA))
)

item_stats_v3 = item_stats_v3.sort_values(
    ["score", "weighted_positive", "pos_count", "item"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

score_dict_v3 = dict(zip(item_stats_v3["item"], item_stats_v3["score"]))

result_v3 = evaluate_popularity_scores(
    score_dict=score_dict_v3,
    model_name=f"Popularity_V3_PosOnly_TimeDecay_{TIME_DECAY_LAMBDA}_Smooth_{ALPHA}"
)

result_v3

,dataset,model,HR@10,NDCG@10,MAP,eval_users
0,MovieLens,Popularity_V3_PosOnly_TimeDecay_0.001_Smooth_10.0,0.494367,0.272668,0.227368,6036


In [50]:
ablation_results = pd.concat([result_v1, result_v2, result_v3], ignore_index=True)
ablation_results

,dataset,model,HR@10,NDCG@10,MAP,eval_users
0,MovieLens,Popularity_V1_PosOnly,0.491716,0.271107,0.226227,6036
1,MovieLens,Popularity_V2_PosOnly_TimeDecay_0.001,0.494367,0.272676,0.227378,6036
2,MovieLens,Popularity_V3_PosOnly_TimeDecay_0.001_Smooth_10.0,0.494367,0.272668,0.227368,6036


## 4. MovieLens：Matrix Factorization


In [ ]:
class ExplicitMF(nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)

    def forward(self, user_idx, item_idx):
        return (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(dim=1)

    @torch.no_grad()
    def score_all_items(self, user_indices, device):
        u = torch.as_tensor(user_indices, dtype=torch.long, device=device)
        user_vecs = self.user_emb(u)
        scores = user_vecs @ self.item_emb.weight.T
        return scores.detach().cpu().numpy()


ml_user_arr = ml_train['user_idx'].to_numpy(dtype=np.int64)
ml_item_arr = ml_train['item_idx'].to_numpy(dtype=np.int64)
ml_rating_arr = ml_train['value'].to_numpy(dtype=np.float32)

ml_model = ExplicitMF(len(ml_user_ids), len(ml_item_ids), dim=64).to(DEVICE)
ml_optimizer = torch.optim.Adam(ml_model.parameters(), lr=1e-3, weight_decay=1e-6)
ml_batch_size = 8192
ml_epochs = 30

for epoch in range(1, ml_epochs + 1):
    ml_model.train()
    perm = np.random.permutation(len(ml_user_arr))
    total_loss = 0.0

    for start in range(0, len(perm), ml_batch_size):
        idx = perm[start:start + ml_batch_size]
        u = torch.as_tensor(ml_user_arr[idx], dtype=torch.long, device=DEVICE)
        i = torch.as_tensor(ml_item_arr[idx], dtype=torch.long, device=DEVICE)
        r = torch.as_tensor(ml_rating_arr[idx], dtype=torch.float32, device=DEVICE)

        pred = ml_model(u, i)
        loss = F.mse_loss(pred, r)

        ml_optimizer.zero_grad()
        loss.backward()
        ml_optimizer.step()

        total_loss += loss.item() * len(idx)

    print(f'Epoch {epoch:02d} | MovieLens MF train MSE = {total_loss / len(perm):.6f}')


Epoch 01 | MovieLens MF train MSE = 14.074127
Epoch 02 | MovieLens MF train MSE = 10.340534
Epoch 03 | MovieLens MF train MSE = 2.338362
Epoch 04 | MovieLens MF train MSE = 1.026803
Epoch 05 | MovieLens MF train MSE = 0.870741
Epoch 06 | MovieLens MF train MSE = 0.825019
Epoch 07 | MovieLens MF train MSE = 0.802263
Epoch 08 | MovieLens MF train MSE = 0.785485
Epoch 09 | MovieLens MF train MSE = 0.771129
Epoch 10 | MovieLens MF train MSE = 0.758745
Epoch 11 | MovieLens MF train MSE = 0.747632
Epoch 12 | MovieLens MF train MSE = 0.737167
Epoch 13 | MovieLens MF train MSE = 0.726919
Epoch 14 | MovieLens MF train MSE = 0.716472
Epoch 15 | MovieLens MF train MSE = 0.706141
Epoch 16 | MovieLens MF train MSE = 0.695692
Epoch 17 | MovieLens MF train MSE = 0.685394
Epoch 18 | MovieLens MF train MSE = 0.675201
Epoch 19 | MovieLens MF train MSE = 0.665089
Epoch 20 | MovieLens MF train MSE = 0.655158
Epoch 21 | MovieLens MF train MSE = 0.645349
Epoch 22 | MovieLens MF train MSE = 0.635619
Epoch 23

In [11]:
ml_model.eval()


def ml_mf_score_fn(user_indices):
    return ml_model.score_all_items(user_indices, DEVICE)

ml_mf_metrics = evaluate_sampled_ranking(
    score_fn=ml_mf_score_fn,
    positive_items=ml_test_positive,
    candidate_items=ml_candidate_items,
    k=K,
)

ml_results = pd.DataFrame([
    {'dataset': 'MovieLens', 'model': 'Popularity', **ml_pop_metrics},
    {'dataset': 'MovieLens', 'model': 'MatrixFactorization', **ml_mf_metrics},
])
ml_results


,dataset,model,HR@10,NDCG@10,MAP,num_eval_users
0,MovieLens,Popularity,0.488985,0.269759,0.225284,6037
1,MovieLens,MatrixFactorization,0.352162,0.188108,0.162951,6037


## 5. Last.fm

In [ ]:
set_seed(SEED)

lastfm = pd.read_csv(BASE_DIR / 'user_artists.dat', sep='\t')
lastfm = lastfm.rename(columns={'userID': 'user', 'artistID': 'item', 'weight': 'value'})
lastfm['user'] = lastfm['user'].astype(np.int64)
lastfm['item'] = lastfm['item'].astype(np.int64)
lastfm['value'] = lastfm['value'].astype(np.float32)

lastfm = make_random_timestamps_for_lastfm(lastfm, seed=SEED)
lastfm = lastfm.sort_values(['user', 'timestamp', 'item']).reset_index(drop=True)

lf_train_parts = []
lf_test_positive_raw = {}
lf_user_full_seen_raw = {}
dropped_short_history = 0

for user, g in lastfm.groupby('user', sort=False):
    g = g.sort_values(['timestamp', 'item']).reset_index(drop=True)
    lf_user_full_seen_raw[int(user)] = set(g['item'].astype(int).tolist())

    if len(g) < 2:
        dropped_short_history += 1
        continue

    train_part = g.iloc[:-1].copy()
    test_row = g.iloc[-1]

    lf_train_parts.append(train_part)
    lf_test_positive_raw[int(user)] = int(test_row['item'])

lf_train_raw = pd.concat(lf_train_parts, ignore_index=True)

lf_user_ids, lf_user2idx, lf_idx2user = build_id_mappings(lf_train_raw['user'])
lf_item_ids, lf_item2idx, lf_idx2item = build_id_mappings(lf_train_raw['item'])

lf_train = lf_train_raw.copy()
lf_train['user_idx'] = encode_series(lf_train['user'], lf_user2idx)
lf_train['item_idx'] = encode_series(lf_train['item'], lf_item2idx)

lf_test_positive = {}
lf_user_full_seen = {}

dropped_user_not_in_train = 0
dropped_unseen_test_item = 0

for u, pos_item in lf_test_positive_raw.items():
    if u not in lf_user2idx:
        dropped_user_not_in_train += 1
        continue

    if pos_item not in lf_item2idx:
        dropped_unseen_test_item += 1
        continue

    u_idx = lf_user2idx[u]
    pos_i_idx = lf_item2idx[pos_item]

    lf_test_positive[u_idx] = pos_i_idx
    lf_user_full_seen[u_idx] = {
        lf_item2idx[i]
        for i in lf_user_full_seen_raw[u]
        if i in lf_item2idx
    }

lf_all_items = np.arange(len(lf_item_ids), dtype=np.int64)
lf_rng = np.random.default_rng(SEED)
lf_candidate_items = {}
users_to_remove = []

for u_idx, pos_i_idx in lf_test_positive.items():
    sampled = sample_99_negatives(
        lf_all_items,
        lf_user_full_seen[u_idx],
        pos_i_idx,
        lf_rng,
        NUM_NEGATIVES
    )
    if sampled is None:
        users_to_remove.append(u_idx)
    else:
        lf_candidate_items[u_idx] = sampled

for u_idx in users_to_remove:
    lf_test_positive.pop(u_idx, None)
    lf_user_full_seen.pop(u_idx, None)

print('Last.fm train interactions =', len(lf_train))
print('Last.fm train users =', lf_train["user_idx"].nunique())
print('Last.fm train items =', lf_train["item_idx"].nunique())
print('Dropped users with too short history =', dropped_short_history)
print('Dropped users not in train mapping =', dropped_user_not_in_train)
print('Dropped users whose test item is unseen in train =', dropped_unseen_test_item)
print('Dropped users due to insufficient negatives =', len(users_to_remove))
print('Final Last.fm eval users =', len(lf_test_positive))

lf_train.head()

Last.fm train interactions = 90942
Last.fm train users = 1884
Last.fm train items = 17394
Dropped users with too short history = 8
Dropped users not in train mapping = 0
Dropped users whose test item is unseen in train = 234
Dropped users due to insufficient negatives = 0
Final Last.fm eval users = 1650


,user,item,value,timestamp,user_idx,item_idx
0,2,85,1638.0,1284452978,0,79
1,2,94,1373.0,1285876913,0,88
2,2,56,6152.0,1292133049,0,50
3,2,51,13883.0,1293280221,0,45
4,2,82,1868.0,1294281501,0,76


## 6. Last.fm：Popularity baseline

In [14]:
lf_pop_scores = np.zeros(len(lf_item_ids), dtype=np.float32)
np.add.at(lf_pop_scores, lf_train['item_idx'].to_numpy(dtype=np.int64), lf_train['value'].to_numpy(dtype=np.float32))


def lf_popularity_score_fn(user_indices):
    return np.repeat(lf_pop_scores[None, :], repeats=len(user_indices), axis=0)

lf_pop_metrics = evaluate_sampled_ranking(
    score_fn=lf_popularity_score_fn,
    positive_items=lf_test_positive,
    candidate_items=lf_candidate_items,
    k=K,
)

pd.DataFrame([{'dataset': 'Last.fm', 'model': 'Popularity', **lf_pop_metrics}])


,dataset,model,HR@10,NDCG@10,MAP,num_eval_users
0,Last.fm,Popularity,0.74303,0.516615,0.456632,1650


## 7. Last.fm：Matrix Factorization

In [ ]:
class BPRMF(nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        self.item_bias = nn.Embedding(num_items, 1)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)
        nn.init.zeros_(self.item_bias.weight)

    def score(self, user_idx, item_idx):
        return (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(dim=1) + self.item_bias(item_idx).squeeze(1)

    @torch.no_grad()
    def score_all_items(self, user_indices, device):
        u = torch.as_tensor(user_indices, dtype=torch.long, device=device)
        user_vecs = self.user_emb(u)
        scores = user_vecs @ self.item_emb.weight.T + self.item_bias.weight.T
        return scores.detach().cpu().numpy()


lf_positive_pairs = lf_train[['user_idx', 'item_idx']].drop_duplicates().to_numpy(dtype=np.int64)
lf_user_pos_sets = {}
for u_idx, i_idx in lf_positive_pairs:
    lf_user_pos_sets.setdefault(int(u_idx), set()).add(int(i_idx))

lf_bpr_model = BPRMF(len(lf_user_ids), len(lf_item_ids), dim=64).to(DEVICE)
lf_optimizer = torch.optim.Adam(lf_bpr_model.parameters(), lr=1e-3, weight_decay=1e-6)
lf_epochs = 15
lf_batch_size = 4096
rng = np.random.default_rng(SEED)

# record training time
start_time = time.time()


for epoch in range(1, lf_epochs + 1):
    lf_bpr_model.train()
    perm = rng.permutation(len(lf_positive_pairs))
    epoch_loss = 0.0
    seen_examples = 0

    for start in range(0, len(perm), lf_batch_size):
        idx = perm[start:start + lf_batch_size]
        batch_pairs = lf_positive_pairs[idx]

        users_np = batch_pairs[:, 0].astype(np.int64)
        pos_np = batch_pairs[:, 1].astype(np.int64)
        neg_np = np.empty(len(batch_pairs), dtype=np.int64)

        for t, u_idx in enumerate(users_np):
            while True:
                j = int(rng.integers(0, len(lf_item_ids)))
                if j not in lf_user_pos_sets[u_idx]:
                    neg_np[t] = j
                    break

        u = torch.as_tensor(users_np, dtype=torch.long, device=DEVICE)
        i = torch.as_tensor(pos_np, dtype=torch.long, device=DEVICE)
        j = torch.as_tensor(neg_np, dtype=torch.long, device=DEVICE)

        pos_scores = lf_bpr_model.score(u, i)
        neg_scores = lf_bpr_model.score(u, j)
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()

        lf_optimizer.zero_grad()
        loss.backward()
        lf_optimizer.step()

        epoch_loss += loss.item() * len(batch_pairs)
        seen_examples += len(batch_pairs)

    print(f'Epoch {epoch:02d} | Last.fm BPR-MF train loss = {epoch_loss / seen_examples:.6f}')

end_time = time.time()
training_time = end_time - start_time
print(f'Total training time: {training_time:.2f} seconds')

Epoch 01 | Last.fm BPR-MF train loss = 0.689977
Epoch 02 | Last.fm BPR-MF train loss = 0.679835
Epoch 03 | Last.fm BPR-MF train loss = 0.670181
Epoch 04 | Last.fm BPR-MF train loss = 0.659939
Epoch 05 | Last.fm BPR-MF train loss = 0.648816
Epoch 06 | Last.fm BPR-MF train loss = 0.635457
Epoch 07 | Last.fm BPR-MF train loss = 0.618418
Epoch 08 | Last.fm BPR-MF train loss = 0.595945
Epoch 09 | Last.fm BPR-MF train loss = 0.565988
Epoch 10 | Last.fm BPR-MF train loss = 0.529291
Epoch 11 | Last.fm BPR-MF train loss = 0.487549
Epoch 12 | Last.fm BPR-MF train loss = 0.445133
Epoch 13 | Last.fm BPR-MF train loss = 0.404010
Epoch 14 | Last.fm BPR-MF train loss = 0.367873
Epoch 15 | Last.fm BPR-MF train loss = 0.337016


In [16]:
lf_bpr_model.eval()


def lf_mf_score_fn(user_indices):
    return lf_bpr_model.score_all_items(user_indices, DEVICE)

lf_mf_metrics = evaluate_sampled_ranking(
    score_fn=lf_mf_score_fn,
    positive_items=lf_test_positive,
    candidate_items=lf_candidate_items,
    k=K,
)

lf_results = pd.DataFrame([
    {'dataset': 'Last.fm', 'model': 'Popularity', **lf_pop_metrics},
    {'dataset': 'Last.fm', 'model': 'MatrixFactorization', **lf_mf_metrics},
])
lf_results


,dataset,model,HR@10,NDCG@10,MAP,num_eval_users
0,Last.fm,Popularity,0.743030,0.516615,0.456632,1650
1,Last.fm,MatrixFactorization,0.789697,0.596087,0.544608,1650
